<a href="https://colab.research.google.com/github/dandrade-io/classification-accusatory-judgments/blob/main/classification-accusatory-judgments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Clasificación de Preguntas Acusatorias en Contratación Pública**

Machine Learning

Proyecto Final



---



  **Integrantes:**


 - Isabella Tulcán

 - Daniel Andrade

 - Jarod Sierra



---




# **1. Introducción**

añadir contexto del problema

# **2. Objetivo**

añadir objetivos del proyecto

# **Instalación de Dependencias**




In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re


#Feauture extraction y preprocesamiento
from sklearn.feature_extraction.text import TfidfVectorizer

#Visualización
from sklearn.manifold import TSNE

# Sentence Transformers (Transfer Learning)
from sentence_transformers import SentenceTransformer


## **1. Carga del Dataset**

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
df = pd.read_excel('dataset.xlsx')
print(f"Forma (filas, columnas): {df.shape}")
print(f"\nNombres de columnas: {list(df.columns)}")
print(f"\nTipos de datos:\n{df.dtypes}")
print(f"\nPrimeras filas:")
df.head()

## **2. Análisis exploratorio de los datos (EDA)**

In [ ]:
#Estadísticas básicas
print(f"Valores nulos por columna:\n{df.isnull().sum()}")
print(f"\nRegistros duplicados (basado en preguntas): {df.duplicated(subset='pregunta').sum()}")
print("\nEstadísticas de variables categóricas:")
print(df.describe(include=['object']))
print("\nValores únicos por columna:")
print(df.nunique())
print(f"\nEstadísticas descriptivas:")
df.describe()


In [ ]:
#Balanceo de variable objetivo
import matplotlib.pyplot as plt
import pandas as pd

# Conteo seguro y ordenado de clases
counts = df['final_pregunta_isAcusatoria'].value_counts().reindex([0, 1], fill_value=0)

# Etiquetas legibles
labels_map = {0: 'No Acusatoria', 1: 'Acusatoria'}
labels = [labels_map[i] for i in counts.index]

# Porcentajes
percentages = (counts / len(df)) * 100

print("\nDistribución de clases:")
print(pd.DataFrame({
    'Clase': labels,
    'Cantidad': counts.values,
    'Porcentaje (%)': percentages.round(2).values
}))

# Crear figura
fig, ax = plt.subplots(figsize=(7, 5))

# Colores
colors = ['#2ecc71', '#e74c3c']

# Gráfico de barras
bars = ax.bar(labels, counts.values, color=colors, edgecolor='black')

max_count = counts.max()

for bar, count, pct in zip(bars, counts.values, percentages.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max_count * 0.05,
        f'{count}\n({pct:.1f}%)',
        ha='center',
        va='bottom',
        fontweight='bold'
    )

ax.set_title('Distribución de clases')
ax.set_ylabel('Cantidad')
ax.set_xlabel('Clase')

# Título general
plt.suptitle('Preguntas\nacusatorias vs no acusatorias', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Resumen numérico
majority_class = labels[counts.values.argmax()]
minority_class = labels[counts.values.argmin()]
ratio = counts.max() / counts.min() if counts.min() > 0 else float('inf')

print(f"\nClase mayoritaria: {majority_class} ({counts.max()} registros, {percentages.max():.2f}%)")
print(f"Clase minoritaria: {minority_class} ({counts.min()} registros, {percentages.min():.2f}%)")
print(f"Ratio de desbalance: {ratio:.2f}:1")

# Interpretación
if percentages.min() < 20:
    print("Conclusión: existe un desbalance fuerte entre clases.")
elif percentages.min() < 40:
    print("Conclusión: existe un desbalance moderado entre clases.")
else:
    print("Conclusión: las clases están relativamente balanceadas.")

In [ ]:
#Distribución de votos de anotadores
fig, ax = plt.subplots(figsize=(12, 6))

# Conteo ordenado
vote_counts = df['sum_pregunta_isAcusatoria'].value_counts().sort_index()

# Porcentajes
percentages = (vote_counts / len(df)) * 100

# Colores según nivel de acuerdo
colors = []
max_votes = vote_counts.index.max()

for v in vote_counts.index:
    if v == 0:
        colors.append('#2ecc71')  # verde
    elif v == max_votes:
        colors.append('#e74c3c')  # rojo
    else:
        colors.append('#f1c40f')  # amarillo

# Gráfico
bars = ax.bar(vote_counts.index, vote_counts.values, color=colors, edgecolor='black')

# Etiquetas
for bar, count, pct in zip(bars, vote_counts.values, percentages.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(vote_counts.values) * 0.02,
        f'{count}\n({pct:.1f}%)',
        ha='center',
        va='bottom',
        fontweight='bold'
    )

# Títulos
ax.set_title('Distribución de votos de anotadores')
ax.set_xlabel('Número de anotadores que marcaron como acusatoria')
ax.set_ylabel('Cantidad de preguntas')

plt.tight_layout()
plt.show()

In [ ]:
#Longitud de texto por clase
df['char_length'] = df['pregunta'].astype(str).str.len()
df['word_count'] = df['pregunta'].astype(str).str.split().str.len()

# Nombres de clase
df['clase_nombre'] = df['final_pregunta_isAcusatoria'].map({
    0: 'No Acusatoria',
    1: 'Acusatoria'
})

# Resumen numérico
summary = df.groupby('clase_nombre')[['char_length', 'word_count']].agg(['mean', 'median', 'std']).round(2)
print("Resumen de longitud por clase:")
print(summary)

fig, axes = plt.subplots(1, 2, figsize=(12,5))

# Histograma
for clase, color in zip(['No Acusatoria', 'Acusatoria'], ['#2ecc71', '#e74c3c']):
    subset = df[df['clase_nombre'] == clase]
    axes[0].hist(subset['word_count'], bins=30, alpha=0.6, label=clase, color=color)

axes[0].set_title('Distribución de longitud (palabras)')
axes[0].legend()

# Promedio
mean_words = df.groupby('clase_nombre')['word_count'].mean()

axes[1].bar(mean_words.index, mean_words.values, color=['#2ecc71', '#e74c3c'])

for i, v in enumerate(mean_words):
    axes[1].text(i, v + 0.2, f'{v:.1f}', ha='center', fontweight='bold')

axes[1].set_title('Promedio de palabras por clase')

plt.tight_layout()
plt.show()


In [ ]:
from collections import Counter

# Asegurarnos de que la columna exista y tratar nulos
texto = df["pregunta"].fillna("").astype(str)

print("Inspección de normalización del texto")

vacios = (texto.str.strip() == "").sum()
print(f"Textos vacíos o con solo espacios: {vacios} / {len(df)}")

mayusculas = texto.str.contains(r"[A-Z]", regex=True).sum()
print(f"Textos con mayúsculas: {mayusculas} / {len(df)}")

tildes = texto.str.contains(r"[áéíóúÁÉÍÓÚ]", regex=True).sum()
print(f"Textos con tildes: {tildes} / {len(df)}")

enie = texto.str.contains(r"[ñÑ]", regex=True).sum()
print(f"Textos con ñ: {enie} / {len(df)}")

numeros = texto.str.contains(r"[0-9]", regex=True).sum()
print(f"Textos con números: {numeros} / {len(df)}")

urls = texto.str.contains(r"http[s]?://|www\.", regex=True).sum()
print(f"Textos con URLs: {urls} / {len(df)}")

signos = texto.str.contains(r"[¿?¡!]", regex=True).sum()
print(f"Textos con signos de interrogación/exclamación: {signos} / {len(df)}")


# Longitud (corregido)
longitud = texto.str.len()

muy_cortas = (longitud < 5).sum()
print(f"\nPreguntas muy cortas (<5 caracteres): {muy_cortas} / {len(df)}")

# Palabras acusatorias
patron_acusatorio = r"\b(?:robo|robó|corrupción|corrupcion|culpable|responsable|coima|soborno|fraude|delito|acusado|acusación|acusacion)\b"

acusatorias_keywords = texto.str.contains(
    patron_acusatorio, case=False, regex=True
).sum()

print(f"Textos con palabras potencialmente acusatorias: {acusatorias_keywords} / {len(df)}")

# Top palabras
print("\nTop 20 palabras más frecuentes")

todas_las_palabras = " ".join(texto).lower().split()
conteo_palabras = Counter(todas_las_palabras)

for palabra, freq in conteo_palabras.most_common(20):
    print(f"{palabra}: {freq}")

## **Conclusiones del análisis exploratorio de los datos (EDA)**

**Conclusiones individuales**

1. El dataset se encuentra limpio a nivel estructural, ya que no presenta valores nulos, lo que permite avanzar directamente al análisis y modelado sin necesidad de aplicar técnicas de imputación.
2. Se observa una baja redundancia en las preguntas, lo cual es positivo para modelos de procesamiento de lenguaje natural, ya que favorece la diversidad de ejemplos y reduce el riesgo de sobreajuste.
3. La variable final_pregunta_isAcusatoria presenta un fuerte desbalance (33:1), lo que implica que métricas como accuracy pueden resultar engañosas. Por ello, será necesario utilizar métricas como precision, recall y F1-score, así como técnicas como class_weight='balanced'.
4. La mayoría de las preguntas no son consideradas acusatorias por los anotadores, lo que confirma la baja frecuencia de esta clase. Asimismo, la presencia de algunos desacuerdos sugiere posibles casos de ambigüedad en la clasificación.
5. Las preguntas acusatorias tienden a ser significativamente más largas, lo que sugiere que incluyen mayor nivel de detalle, contexto o argumentación.
6. El texto no se encuentra completamente normalizado, presentando variabilidad en el uso de mayúsculas, tildes y signos, por lo que será necesario aplicar preprocesamiento adecuado.
7. Las preguntas acusatorias no se identifican únicamente mediante palabras clave, sino que dependen del contexto y el tono, lo que sugiere la necesidad de modelos que capturen información semántica más compleja.

**Conclusión general:**

El análisis exploratorio revela que el dataset presenta una alta diversidad textual y ausencia de valores nulos, lo que facilita su uso para modelado. Sin embargo, se identifica un fuerte desbalance en la variable objetivo, donde las preguntas acusatorias representan apenas el 2.94% del total. Este desbalance implica la necesidad de utilizar métricas robustas y técnicas específicas durante el entrenamiento del modelo.

Adicionalmente, se observa que las preguntas acusatorias tienden a ser significativamente más largas que las no acusatorias, lo que sugiere diferencias en la estructura y complejidad del lenguaje utilizado. Este hallazgo indica que la longitud del texto puede ser una variable relevante para la clasificación.

Finalmente, el análisis textual muestra que la identificación de preguntas acusatorias no depende únicamente de palabras clave, sino del contexto general de la pregunta, lo que sugiere la necesidad de aplicar técnicas de procesamiento de lenguaje natural más avanzadas.





## **3. Preprocesamiento del texto**

In [ ]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

#Stopwords en español

spanish_stopwords = set(stopwords.words('spanish'))
custom_stopwords = spanish_stopwords - {'no', 'sí', 'si'}

#Función de preprocesamiento

def texto_preprocesado(text):
    text = str(text).lower()

    # Remover URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # Mantener letras en español, números y espacios
    text = re.sub(r'[^a-záéíóúñü0-9\s]', ' ', text)

    # Normalizar espacios
    text = re.sub(r'\s+', ' ', text).strip()

    # Tokenización simple
    tokens = text.split()

    # Eliminar stopwords y tokens muy cortos
    tokens = [t for t in tokens if t not in custom_stopwords and len(t) > 2]

    return ' '.join(tokens)

#Aplicar preprocesamiento
df['pregunta_limpia'] = df['pregunta'].apply(texto_preprocesado)


#Ejemplos

print("Ejemplos de preprocesamiento:\n")

sample_examples = df[['pregunta', 'pregunta_limpia']].sample(5, random_state=RANDOM_STATE)

for i, row in enumerate(sample_examples.itertuples(index=False), 1):
    print(f"Ejemplo {i}")
    print(f"Original: {row.pregunta[:250]}")
    print(f"Limpio:   {row.pregunta_limpia[:250]}")
    print("-" * 80)



train_test_split divide los datos en conjuntos de entrenamiento y prueba, manteniendo la proporción de clases mediante stratify y garantizando reproducibilidad mediante random_state.

In [ ]:
#Definir variables
X_clean = df['pregunta_limpia']  # Para modelos clásicos: Bag of Words, TF-IDF, etc.
X_raw = df['pregunta']           # Para transformers / embeddings
y = df['final_pregunta_isAcusatoria']

#Split estratificado
X_train_clean, X_test_clean, X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_clean,
    X_raw,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE, #se usa siempre el mismo shuffle
    stratify=y #hace que el split respete la proporción de clases, se basa en y ( final_pregunta_isAcusatoria)
)

#Verificación del split
print("\nTamaño de conjuntos:")
print(f"Train: {len(X_train_clean)} muestras")
print(f"Test:  {len(X_test_clean)} muestras")

print("\nDistribución de clases en Train:")
print(y_train.value_counts())
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nDistribución de clases en Test:")
print(y_test.value_counts())
print((y_test.value_counts(normalize=True) * 100).round(2))


#Revisión rápida de textos vacíos tras limpieza
empty_clean_train = (X_train_clean.str.strip() == '').sum()
empty_clean_test = (X_test_clean.str.strip() == '').sum()

print("\nTextos vacíos después del preprocesamiento:")
print(f"Train vacíos: {empty_clean_train}")
print(f"Test vacíos:  {empty_clean_test}")

## **4. Feature Extraction**

En esta sección se comparan dos enfoques para la extracción de característcas a partir del texto, con el objetivo de evaluar cuál representa mejor la información para el problema de clasificación de preguntas acusatorias.

Por un lado, se utilizará TF-IDF (Term Frequency – Inverse Document Frequency), un método clásico basado en la frecuencia de términos, que transforma el texto en vectores dispersos (sparse) donde cada dimensión representa la importancia de una palabra dentro del corpus.

Por otro lado, se emplea Sentence Transformers, un enfoque basado en transfer learning que utiliza modelos preentrenados para generar representaciones densas (embeddings) del texto, capturando no solo la presencia de palabras, sino también su contexto y significado semántico.

Para comparar ambos enfoques, se evaluará el desempeño de modelos entrenados con cada representación utilizando Repeated Stratified K-Fold Cross Validation, lo cual permite obtener estimaciones más robustas del rendimiento manteniendo la proporción de clases en cada partición. Como métrica principal se utilizará el F1-score (macro), dado el fuerte desbalance presente en la variable objetivo.

Finalmente, se aplicará el test estadístico de Wilcoxon sobre los resultados obtenidos, con el fin de determinar si las diferencias de desempeño entre ambos enfoques son estadísticamente significativas.

**Nota:** No se realiza un proceso adicional de feature selection, ya que en problemas de texto, especialmente al utilizar técnicas como TF-IDF o embeddings preentrenados, la representación de las características ya implica una transformación relevante del espacio original.

P
En esta etapa se construyen dos pipelines de extracción de características y clasificación, con el objetivo de comparar dos formas distintas de representar el texto para la detección de preguntas acusatorias.

El primer pipeline utiliza TF-IDF junto con Regresión Logística. TF-IDF transforma cada pregunta en un vector numérico basado en la importancia de las palabras dentro del corpus. Se configuró con max_features=5000 para limitar la dimensionalidad y evitar un espacio excesivamente grande de características, y con ngram_range=(1,2) para incluir tanto unigramas como bigramas, ya que en este problema ciertas combinaciones de palabras pueden aportar más contexto que palabras aisladas. Además, se usa sublinear_tf=True para suavizar el impacto de términos muy frecuentes. Como clasificador se emplea Regresión Logística con class_weight='balanced', debido al fuerte desbalance detectado en la variable objetivo.

El segundo pipeline utiliza Sentence Transformers para generar embeddings densos del texto. A diferencia de TF-IDF, este enfoque no se basa únicamente en frecuencia de palabras, sino que intenta capturar información semántica y contextual. Esto resulta especialmente relevante en este problema, ya que el EDA mostró que las preguntas acusatorias no se distinguen únicamente por palabras clave, sino también por el tono y el contexto. Posteriormente, estos embeddings se clasifican con una Regresión Logística, nuevamente con class_weight='balanced' para compensar el desbalance de clases.

Se decidió no incluir SMOTE en esta primera versión. En el caso de TF-IDF, esto se debe a que sus representaciones son dispersas y de muy alta dimensión, por lo que generar ejemplos sintéticos no siempre resulta adecuado ni interpretable. En el caso de embeddings, aunque SMOTE podría ser más razonable, primero se busca una comparación base limpia entre ambas representaciones usando el mismo clasificador y la misma estrategia de balanceo. De esta manera, los resultados serán más fáciles de interpretar y atribuir a la representación del texto, no al método de remuestreo.

En conjunto, estos dos pipelines permiten comparar un enfoque clásico basado en frecuencia de términos frente a un enfoque moderno basado en transfer learning, manteniendo constante el modelo de clasificación para que la diferencia observada provenga principalmente de la forma en que se representa el texto.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sentence_transformers import SentenceTransformer

# 4.2 Pipeline TF-IDF + Logistic Regression

tfidf_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=3000,
        ngram_range=(1, 1),
        sublinear_tf=True
    )),
    ('clf', LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        solver='liblinear',
        random_state=RANDOM_STATE
    ))
])

print("Pipeline TF-IDF:")
print(tfidf_pipeline)


# 4.3 Transformer personalizado para embeddings
class SentenceTransformerEncoder(BaseEstimator, TransformerMixin):
    """
    Wrapper de SentenceTransformer compatible con sklearn Pipeline.
    Convierte una lista de textos en embeddings densos.
    """

    def __init__(self, model_name='paraphrase-multilingual-MiniLM-L12-v2', batch_size=64):
        self.model_name = model_name
        self.batch_size = batch_size

    def fit(self, X, y=None):
        self.model_ = SentenceTransformer(self.model_name)
        return self

    def transform(self, X):
        texts = X.tolist() if hasattr(X, 'tolist') else list(X)
        return self.model_.encode(
            texts,
            show_progress_bar=False,
            batch_size=self.batch_size
        )

# 4.4 Pipeline Embeddings + Logistic Regression
embedding_pipeline = Pipeline([
    ('encoder', SentenceTransformerEncoder(
        model_name='paraphrase-multilingual-MiniLM-L12-v2', #Es un modelo: multilingüe, optimizado para entender similitud entre frases, ligero y rápido
        batch_size=64
    )),
    ('clf', LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        solver='liblinear', #algoritmo interno de aprendizaje. Funciona bien con clasif. binaria, datasets peq-med, problemas de desbalance.
        random_state=RANDOM_STATE
    ))
])

print("\nPipeline Embeddings:")
print(embedding_pipeline)

Se definieron dos pipelines para el modelado: uno basado en TF-IDF y otro en embeddings generados con Sentence Transformers. Ambos utilizan Regresión Logística como clasificador y aplican balanceo de clases mediante class_weight='balanced'. La diferencia entre ambos radica en la forma en que representan el texto.

No se empleó SMOTE en esta etapa inicial, ya que en representaciones como TF-IDF, caracterizadas por alta dimensionalidad y dispersión, la generación de ejemplos sintéticos puede no ser adecuada. Además, se utilizó class_weight='balanced' para manejar el desbalance de clases. Esto permite mantener una comparación más limpia entre los enfoques de representación del texto.

### **Comparación: Repeated Stratified K-Fold CV**

In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
import numpy as np
import time

cv_feat = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=2,
    random_state=RANDOM_STATE
)

# TF-IDF
print("Evaluación TF-IDF pipeline...")
start = time.time()

scores_tfidf = cross_val_score(
    tfidf_pipeline,
    X_train_clean,
    y_train,
    cv=cv_feat,
    scoring='f1_macro',
    n_jobs=1
)

end = time.time()
print(f"TF-IDF F1 Macro: {scores_tfidf.mean():.4f} (+/- {scores_tfidf.std():.4f})")
print(f"Tiempo TF-IDF: {end - start:.2f} segundos")

# Embeddings
print("\nEvaluación Embeddings pipeline...")
start = time.time()

scores_embed = cross_val_score(
    embedding_pipeline,
    X_train_raw,
    y_train,
    cv=cv_feat,
    scoring='f1_macro',
    n_jobs=1
)

end = time.time()
print(f"Embeddings F1 Macro: {scores_embed.mean():.4f} (+/- {scores_embed.std():.4f})")
print(f"Tiempo Embeddings: {end - start:.2f} segundos")

# Comparación
print("\nComparación de modelos:")
print(f"TF-IDF      → Media: {np.mean(scores_tfidf):.4f}, Std: {np.std(scores_tfidf):.4f}")
print(f"Embeddings  → Media: {np.mean(scores_embed):.4f}, Std: {np.std(scores_embed):.4f}")

In [ ]:
#Test de Wilcoxon

from scipy.stats import wilcoxon

# Verificación básica

print("Número de scores TF-IDF:", len(scores_tfidf))
print("Número de scores Embeddings:", len(scores_embed))

if len(scores_tfidf) != len(scores_embed):
    raise ValueError("Los vectores de scores no tienen la misma longitud. No se puede aplicar Wilcoxon pareado.")


# Estadísticos descriptivos

alpha = 0.05

mean_tfidf = np.mean(scores_tfidf)
std_tfidf = np.std(scores_tfidf)

mean_embed = np.mean(scores_embed)
std_embed = np.std(scores_embed)

diff = scores_tfidf - scores_embed
mean_diff = np.mean(diff)


# Test de Wilcoxon

stat_feat, p_value_feat = wilcoxon(scores_tfidf, scores_embed)


# Tabla resumen

summary_df = pd.DataFrame({
    'Modelo': ['TF-IDF', 'Sentence Transformers'],
    'Media F1 Macro': [mean_tfidf, mean_embed],
    'Desviación estándar': [std_tfidf, std_embed]
})

print("\nResumen de resultados:")
print(summary_df.round(4))

print("\nDiferencias por fold (TF-IDF - Embeddings):")
for i, d in enumerate(diff, start=1):
    print(f"Fold {i}: {d:.4f}")

print("\nTest de Wilcoxon: TF-IDF vs Sentence Transformers")
print(f"Estadístico W: {stat_feat:.4f}")
print(f"p-value: {p_value_feat:.6f}")
print(f"Nivel de significancia (alpha): {alpha}")

print(f"\nDiferencia media (TF-IDF - Embeddings): {mean_diff:.4f}")


# Interpretación automática

print("\nInterpretación:")

if mean_tfidf > mean_embed:
    print(f"TF-IDF presenta un mejor desempeño promedio que Sentence Transformers por {mean_diff:.4f} puntos de F1 macro.")
else:
    print(f"Sentence Transformers presenta un mejor desempeño promedio que TF-IDF por {abs(mean_diff):.4f} puntos de F1 macro.")

if p_value_feat < alpha:
    print("La diferencia observada es estadísticamente significativa, por lo que se rechaza la hipótesis nula.")
else:
    print("La diferencia observada no es estadísticamente significativa, por lo que no se rechaza la hipótesis nula.")

#
# Visualización 1: Diferencia por fold
#
fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.bar(range(1, len(diff) + 1), diff, edgecolor='black')

# Colorear según signo de la diferencia
for bar, value in zip(bars, diff):
    if value >= 0:
        bar.set_facecolor('#3498db')   # azul si TF-IDF gana
    else:
        bar.set_facecolor('#e67e22')   # naranja si Embeddings gana

ax.axhline(0, color='black', linestyle='--', linewidth=1)
ax.set_title('Diferencia de rendimiento por fold\n(TF-IDF - Sentence Transformers)')
ax.set_xlabel('Fold')
ax.set_ylabel('Diferencia en F1 Macro')

for i, value in enumerate(diff, start=1):
    ax.text(i, value + (0.002 if value >= 0 else -0.005), f'{value:.3f}',
            ha='center', va='bottom' if value >= 0 else 'top', fontsize=9)

plt.tight_layout()
plt.show()

#
# Visualización 2: Comparación punto a punto
#
fig, ax = plt.subplots(figsize=(6, 6))

ax.scatter(scores_embed, scores_tfidf, s=80, edgecolor='black')

# Línea de igualdad
min_val = min(scores_embed.min(), scores_tfidf.min()) - 0.02
max_val = max(scores_embed.max(), scores_tfidf.max()) + 0.02
ax.plot([min_val, max_val], [min_val, max_val], linestyle='--', color='gray')

# Etiquetas de cada fold
for i, (x, y) in enumerate(zip(scores_embed, scores_tfidf), start=1):
    ax.text(x + 0.002, y + 0.002, f'F{i}', fontsize=9)

ax.set_xlim(min_val, max_val)
ax.set_ylim(min_val, max_val)
ax.set_xlabel('Sentence Transformers (F1 Macro)')
ax.set_ylabel('TF-IDF (F1 Macro)')
ax.set_title('Comparación punto a punto entre métodos')

plt.tight_layout()
plt.show()

## **Interpretación de los resultados**

Los resultados obtenidos muestran que el modelo basado en TF-IDF alcanza un F1-score macro promedio de 0.7038, superando al modelo basado en Sentence Transformers, que obtiene un valor de 0.6346. Esto representa una diferencia promedio de 0.0692 puntos, lo cual es considerable en términos de desempeño en tareas de clasificación.

El análisis por fold evidencia que TF-IDF supera a Sentence Transformers en 9 de las 10 particiones evaluadas, lo que indica que la mejora observada es consistente y no producto del azar. Aunque el modelo basado en embeddings presenta una menor desviación estándar, lo que sugiere mayor estabilidad, su desempeño promedio es inferior.

Adicionalmente, el test de Wilcoxon arrojó un valor-p de 0.003906, inferior al nivel de significancia establecido (α = 0.05), lo que permite rechazar la hipótesis nula. Esto indica que la diferencia entre ambos métodos de extracción de características es estadísticamente significativa.

En conjunto, estos resultados sugieren que el enfoque basado en TF-IDF es más adecuado para este problema, posiblemente debido a que la distinción entre preguntas acusatorias y no acusatorias depende en mayor medida de patrones léxicos y combinaciones específicas de palabras, las cuales son capturadas de manera efectiva por este método.